# Bayesian Level-Space GMSL Model

Self-consistent Bayesian calibration of the polynomial rate-temperature relationship for global mean sea level rise, fitting cumulative sea level directly without intermediate kinematics.

**Generative model:**
$$
\text{rate}(t) = a\,T(t)^2 + b\,T(t) + c
$$
$$
H(t) = H_0 + \int_0^t \text{rate}(\tau)\,d\tau
$$
$$
H_{\text{obs}}(t) \sim \mathcal{N}\bigl(H(t),\; \sigma(t)^2\bigr)
$$
where $\sigma(t)^2 = \sigma_{\text{gmsl}}(t)^2 + \sigma_{\text{TWS}}(t)^2 + \sigma_{\text{extra}}^2$ combines time-varying formal uncertainty, structural TWS uncertainty, and a learned model-inadequacy term.

**Advantages over previous approaches:**
- No intermediate kinematics step (avoids kernel-overlap correlation)
- No FGLS or nugget corrections
- Observations are genuinely independent (annual GMSL)
- Forward model embeds the physics (integration of rate polynomial)
- Time-varying uncertainty properly down-weights less reliable data
- Exponential PC prior on $a$ shrinks toward simpler order-1 model; half-normal on $b$ enforces $b \geq 0$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from pathlib import Path

from bayesian_dols import (
    build_level_design_vectors,
    fit_bayesian_level,
    BayesianLevelResult,
    check_convergence,
    build_dols_design_matrix,
    calibrate_exponential_prior_a,
    prior_predictive_rate_check,
)
from slr_analysis import calibrate_dols, resample_to_monthly
from slr_projections import project_gmsl_ensemble

# Constants
M_TO_MM = 1000.0
BASELINE_YEAR = 2005.0
N_SAMPLES = 3000
N_WALKERS = 32
N_BURNIN = 1000

# Paths
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
H5_PATH = DATA_DIR / 'processed' / 'slr_processed_data.h5'
FIG_DIR = BASE_DIR / 'figures'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print(f'Constants: BASELINE_YEAR={BASELINE_YEAR}, '
      f'N_SAMPLES={N_SAMPLES}, N_WALKERS={N_WALKERS}')

## 1. Data Loading

Load Frederikse et al. (2020) GMSL reconstruction and Berkeley Earth global mean surface temperature from the pre-built HDF5 store. All series are baseline-harmonized to 1995–2005.

In [ ]:
# ============================================================
# Load data from HDF5
# ============================================================
with pd.HDFStore(str(H5_PATH), 'r') as store:
    config = store['/config']
    df_frederikse = store['/harmonized/df_frederikse_h']
    df_berkeley = store['/harmonized/df_berkeley_h']

    # Temperature projections for forward runs
    temp_projections = {}
    ssp_keys = {
        'SSP1-2.6': 'SSP1_2_6', 'SSP2-4.5': 'SSP2_4_5',
        'SSP3-7.0': 'SSP3_7_0', 'SSP5-8.5': 'SSP5_8_5',
    }
    for name, key in ssp_keys.items():
        temp_projections[name] = store[f'/projections/temp/{key}']

    # IPCC GMSL for comparison
    ipcc_gmsl = {}
    ssp_gmsl_keys = {
        'SSP1-2.6': 'ssp126', 'SSP2-4.5': 'ssp245',
        'SSP3-7.0': 'ssp370', 'SSP5-8.5': 'ssp585',
    }
    for name, key in ssp_gmsl_keys.items():
        ipcc_gmsl[name] = store[f'/projections/gmsl/{key}']

# Extract GMSL observations
H_obs_full = df_frederikse['gmsl'].values            # meters
sigma_gmsl = df_frederikse['gmsl_sigma'].values      # meters
tws_sigma = df_frederikse['tws_sigma'].values        # meters
obs_years = np.array([
    t.year + (t.month - 0.5) / 12 for t in df_frederikse.index
])

# Monthly temperature
T_monthly = df_berkeley['temperature'].values
time_monthly = np.array([
    t.year + (t.month - 0.5) / 12 for t in df_berkeley.index
])

print(f'Frederikse GMSL: {len(H_obs_full)} annual obs, '
      f'{obs_years[0]:.1f}–{obs_years[-1]:.1f}')
print(f'Berkeley Earth T: {len(T_monthly)} monthly obs, '
      f'{time_monthly[0]:.1f}–{time_monthly[-1]:.1f}')
print(f'GMSL range: {H_obs_full.min()*M_TO_MM:.1f} to '
      f'{H_obs_full.max()*M_TO_MM:.1f} mm')
print(f'T range: {T_monthly.min():.2f} to {T_monthly.max():.2f} \u00b0C')

## 2. Observation Uncertainty Model

The total observation uncertainty combines three components:
$$
\sigma(t)^2 = \sigma_{\text{gmsl}}(t)^2 + \sigma_{\text{TWS}}(t)^2 + \sigma_{\text{extra}}^2
$$

- $\sigma_{\text{gmsl}}(t)$: Formal GMSL uncertainty from Frederikse et al. (19→5 mm, declining 1900→2018)
- $\sigma_{\text{TWS}}(t)$: Structural TWS uncertainty = Frederikse `tws_sigma` + dam-filling era inflation
- $\sigma_{\text{extra}}$: Sampled parameter capturing model inadequacy (ENSO, volcanism, unmodeled processes)

The dam-filling era (~1950–2000) saw large reservoir impoundment that transferred water from land to ocean. Hydrological models used to estimate TWS before GRACE (~2002) have significant structural uncertainty that the formal `tws_sigma` may understate.

In [ ]:
# ============================================================
# Construct sigma_TWS(t) with dam-filling structural uncertainty
# ============================================================
# Dam-filling era: Gaussian hump centred ~1970, width ~25 yr
DAM_PEAK = 1970.0
DAM_WIDTH = 25.0       # years (1-sigma of Gaussian)
DAM_AMPLITUDE = 0.003  # 3 mm structural uncertainty (meters)

dam_extra = DAM_AMPLITUDE * np.exp(
    -0.5 * ((obs_years - DAM_PEAK) / DAM_WIDTH)**2
)
sigma_tws_total = np.sqrt(tws_sigma**2 + dam_extra**2)

# Total fixed observation uncertainty (sigma_extra added inside likelihood)
sigma_obs_fixed = np.sqrt(sigma_gmsl**2 + sigma_tws_total**2)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(obs_years, 0, sigma_gmsl * M_TO_MM,
                alpha=0.3, label=r'$\sigma_{\rm gmsl}$ (Frederikse)')
ax.plot(obs_years, tws_sigma * M_TO_MM, 'g--',
        label=r'$\sigma_{\rm TWS}$ (Frederikse)', lw=1.5)
ax.plot(obs_years, dam_extra * M_TO_MM, 'r:',
        label='Dam-filling structural', lw=1.5)
ax.plot(obs_years, sigma_tws_total * M_TO_MM, 'g-',
        label=r'$\sigma_{\rm TWS}$ total', lw=2)
ax.plot(obs_years, sigma_obs_fixed * M_TO_MM, 'k-',
        label=r'$\sigma_{\rm obs}$ (excl $\sigma_{\rm extra}$)', lw=2)
ax.set_xlabel('Year')
ax.set_ylabel('Uncertainty (mm)')
ax.set_title('Observation uncertainty components')
ax.legend(loc='upper right')
ax.set_xlim(1900, 2020)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

print(f'sigma_obs range: {sigma_obs_fixed.min()*M_TO_MM:.1f} to '
      f'{sigma_obs_fixed.max()*M_TO_MM:.1f} mm')

## 3. Forward Model Design

The forward model integrates the rate polynomial on the **monthly** temperature grid (capturing sub-annual variability in $T^2$), then extracts predictions at the 119 annual observation times.

Because the model is linear in the parameters $(a, b, c, H_0)$:
$$
H_{\rm model}(t) = a \cdot I_2(t) + b \cdot I_1(t) + c \cdot I_0(t) + H_0
$$
where $I_2 = \int T^2\,d\tau$, $I_1 = \int T\,d\tau$, $I_0 = t - t_0$, we pre-compute these integrals once. Each MCMC log-probability evaluation is then $O(n_{\rm obs}) = O(119)$.

In [ ]:
# ============================================================
# Build design vectors on the monthly temperature grid
# ============================================================
# Restrict Berkeley Earth to the Frederikse observation range
mask_berk = (time_monthly >= obs_years[0]) & (time_monthly <= obs_years[-1])
T_monthly_fit = T_monthly[mask_berk]
time_monthly_fit = time_monthly[mask_berk]

dv = build_level_design_vectors(
    temperature_monthly=T_monthly_fit,
    time_monthly=time_monthly_fit,
    obs_times=obs_years,
)

I2_obs = dv['I2_obs']
I1_obs = dv['I1_obs']
I0_obs = dv['I0_obs']

print(f'Monthly grid: {len(T_monthly_fit)} points '
      f'({time_monthly_fit[0]:.1f}–{time_monthly_fit[-1]:.1f})')
print(f'Observation indices: {len(dv["obs_idx"])} annual points')
print(f'\nIntegral ranges:')
print(f'  I2 (∫T²):  {I2_obs[0]:.3f} to {I2_obs[-1]:.3f}')
print(f'  I1 (∫T):   {I1_obs[0]:.3f} to {I1_obs[-1]:.3f}')
print(f'  I0 (time):  {I0_obs[0]:.3f} to {I0_obs[-1]:.3f} yr')

# Verify against DOLS design matrix for consistency
df_fred_m = resample_to_monthly(
    df_frederikse, value_col='gmsl', unc_col='gmsl_sigma'
)
common = df_fred_m.index.intersection(df_berkeley.index)
sl_s = df_fred_m.loc[common, 'gmsl']
temp_s = df_berkeley.loc[common, 'temperature']
dols_dm = build_dols_design_matrix(sl_s, temp_s, order=2, n_lags=0)

# Compare at overlapping annual times
dols_I2 = dols_dm['X'][:, 0]   # first column = integral of T^2
dols_I1 = dols_dm['X'][:, 1]   # second column = integral of T
dols_I0 = dols_dm['X'][:, 2]   # third column = trend (time)
print(f'\nDOLS design matrix comparison (n_lags=0):')
print(f'  DOLS n_obs = {len(dols_I2)}, Bayesian n_obs = {len(I2_obs)}')
# The DOLS matrix may have different length due to NaN handling;
# compare the endpoint values as a sanity check
print(f'  DOLS I2 endpoint: {dols_I2[-1]:.3f}, Bayesian: {I2_obs[-1]:.3f}')
print(f'  DOLS I1 endpoint: {dols_I1[-1]:.3f}, Bayesian: {I1_obs[-1]:.3f}')
print(f'  DOLS I0 endpoint: {dols_I0[-1]:.3f}, Bayesian: {I0_obs[-1]:.3f}')

# Collinearity check
from numpy import corrcoef
rho_21 = corrcoef(I2_obs, I1_obs)[0, 1]
rho_20 = corrcoef(I2_obs, I0_obs)[0, 1]
rho_10 = corrcoef(I1_obs, I0_obs)[0, 1]
print(f'\nCollinearity:')
print(f'  corr(I2, I1) = {rho_21:.4f}')
print(f'  corr(I2, I0) = {rho_20:.4f}')
print(f'  corr(I1, I0) = {rho_10:.4f}')
print('  (High collinearity expected — priors will constrain the posterior)')

## 4. Model Specification

### Priors

| Parameter | Prior | Physical motivation |
|-----------|-------|--------------------|
| $a$ (d$\alpha$/dT) | Exponential(mean = 2.2 mm/yr/$\degree$C$^2$) | PC prior: mode at $a=0$, shrinks toward order-1; calibrated via $P(a>5)=0.10$ |
| $b$ ($\alpha_0$) | HalfNormal($\sigma$ = 10 mm/yr/$\degree$C) | Warming raises sea level; $b \geq 0$ |
| $c$ (trend) | Normal(2.0, 5.0 mm/yr) | Rate at T=0; baseline-dependent, unconstrained |
| $\sigma_{\rm extra}$ | HalfCauchy(0, 5 mm) | Model inadequacy; heavy tail allows large values |
| $H_0$ | Normal($H_{\rm obs}(t_0)$, 50 mm) | Sea level at first observation |

The **Exponential (PC) prior** on $a$ implements a penalised-complexity principle: the quadratic model ($a > 0$) is a complexity extension of the simpler linear model ($a = 0$). The prior has its mode at zero and decays monotonically, providing genuine shrinkage toward the simpler model. The calibration $P(a > 5\;\text{mm/yr/}\degree\text{C}^2) = 0.10$ yields mean $\mu_a = 2.17\;\text{mm/yr/}\degree\text{C}^2$.

### Likelihood

Independent Gaussian:
$$
\log \mathcal{L} = -\frac{1}{2} \sum_i \left[ \frac{(H_{\rm obs,i} - H_{\rm model,i})^2}{\sigma_i^2} + 2\log\sigma_i \right]
$$
where $\sigma_i^2 = \sigma_{\rm obs}(t_i)^2 + \sigma_{\rm extra}^2$.

In [ ]:
# ============================================================
# Prior configuration (all in meters)
# ============================================================
# Exponential (PC) prior on a: P(a > 5 mm/yr/°C²) = 0.10
PRIOR_SCALE_A = calibrate_exponential_prior_a(
    prob_exceed=0.10, a_threshold=0.005  # 5 mm/yr/°C² in meters
)
PRIOR_SCALE_B = 0.010          # HalfNormal sigma for b (10 mm/yr/°C)
PRIOR_C_MEAN = 0.002           # Normal mean for c (2 mm/yr)
PRIOR_C_SIGMA = 0.005          # Normal sigma for c (5 mm/yr)
PRIOR_SIGMA_EXTRA_SCALE = 0.005  # HalfCauchy scale (5 mm)
PRIOR_H0_SIGMA = 0.050         # Normal sigma for H0 (50 mm)

print('=== Prior Summary ===')
print(f'  a (d\u03b1/dT): Exp(mean = {PRIOR_SCALE_A*M_TO_MM:.2f} mm/yr/\u00b0C\u00b2)')
print(f'     Tail: P(a > 5 mm/yr/\u00b0C\u00b2) = 0.10')
print(f'  b (\u03b1\u2080):    HalfNormal(\u03c3 = {PRIOR_SCALE_B*M_TO_MM:.1f} mm/yr/\u00b0C)')
print(f'  c (trend): Normal({PRIOR_C_MEAN*M_TO_MM:.1f}, {PRIOR_C_SIGMA*M_TO_MM:.1f} mm/yr)')
print(f'  \u03c3_extra:  HalfCauchy(0, {PRIOR_SIGMA_EXTRA_SCALE*M_TO_MM:.1f} mm)')
print(f'  H\u2080:       Normal(H_obs[0], {PRIOR_H0_SIGMA*M_TO_MM:.1f} mm)')

# Prior predictive check
ppc = prior_predictive_rate_check(
    prior_scale_a=PRIOR_SCALE_A,
    prior_scale_b=PRIOR_SCALE_B,
    prior_c_mean=PRIOR_C_MEAN,
    prior_c_sigma=PRIOR_C_SIGMA,
    seed=42,
)
print('\n=== Prior Predictive Rate Check (mm/yr) ===')
print(f'{"T (\u00b0C)":<8} {"Median":>8} {"5%":>8} {"95%":>8} {"99%":>8}')
print('-' * 40)
for T_val, stats in ppc.items():
    print(f'{T_val:<8.1f} {stats["median"]*M_TO_MM:8.1f} '
          f'{stats["p5"]*M_TO_MM:8.1f} {stats["p95"]*M_TO_MM:8.1f} '
          f'{stats["p99"]*M_TO_MM:8.1f}')

## 5. MCMC Sampling

In [ ]:
# ============================================================
# Run Bayesian level-space fit
# ============================================================
result = fit_bayesian_level(
    H_obs=H_obs_full,
    sigma_obs=sigma_obs_fixed,
    I2_obs=I2_obs,
    I1_obs=I1_obs,
    I0_obs=I0_obs,
    prior_scale_a=PRIOR_SCALE_A,
    prior_scale_b=PRIOR_SCALE_B,
    prior_c_mean=PRIOR_C_MEAN,
    prior_c_sigma=PRIOR_C_SIGMA,
    prior_sigma_extra_scale=PRIOR_SIGMA_EXTRA_SCALE,
    prior_H0_sigma=PRIOR_H0_SIGMA,
    n_samples=N_SAMPLES,
    n_walkers=N_WALKERS,
    n_burnin=N_BURNIN,
    progress=True,
    seed=42,
)

# Summary table
print('\n' + '=' * 70)
print('Posterior Summary (mm/yr)')
print('=' * 70)
names_phys = ['d\u03b1/dT', '\u03b1\u2080', 'trend']
print(f'{"Parameter":<12} {"Mean":>10} {"94% HDI":>24}')
print('-' * 50)
for k, name in enumerate(names_phys):
    v = result.physical_coefficients[k] * M_TO_MM
    hdi = result.physical_hdi_94[k] * M_TO_MM
    print(f'{name:<12} {v:10.3f}      [{hdi[0]:.3f}, {hdi[1]:.3f}]')

print(f'\n\u03c3_extra:  {np.median(result.sigma_extra_posterior)*M_TO_MM:.2f} mm '
      f'[{np.percentile(result.sigma_extra_posterior, 3)*M_TO_MM:.2f}, '
      f'{np.percentile(result.sigma_extra_posterior, 97)*M_TO_MM:.2f}]')
print(f'R\u00b2 = {result.r2:.4f}')

## 6. Posterior Diagnostics

Trace plots verify mixing, corner plots reveal parameter correlations (especially the expected $a$–$b$ trade-off from ∫T²/∫T collinearity), and autocorrelation plots check effective sample sizes.

In [ ]:
# ============================================================
# Trace plots
# ============================================================
fig, axes = plt.subplots(5, 2, figsize=(14, 12))
param_labels = [r'$a$ (d$\alpha$/dT)', r'$b$ ($\alpha_0$)', r'$c$ (trend)',
                r'$\log\sigma_{\rm extra}$', r'$H_0$']
chain_full = result.trace.posterior

for i, (name, label) in enumerate(zip(
    ['dalpha_dT', 'alpha0', 'trend', 'log_sigma_extra', 'H0'],
    param_labels)):
    data = chain_full[name].values  # (n_chains, n_steps)
    scale = M_TO_MM if i < 3 else (M_TO_MM if i == 4 else 1.0)

    # Trace
    ax = axes[i, 0]
    for ch in range(data.shape[0]):
        ax.plot(data[ch] * scale, alpha=0.4, lw=0.5)
    ax.set_ylabel(label)
    if i == 0:
        ax.set_title('Trace')

    # Marginal posterior
    ax = axes[i, 1]
    for ch in range(data.shape[0]):
        ax.hist(data[ch].ravel() * scale, bins=50, alpha=0.3, density=True)
    ax.set_ylabel('Density')
    if i == 0:
        ax.set_title('Marginal posterior')

axes[-1, 0].set_xlabel('MCMC step')
axes[-1, 1].set_xlabel('Value')
plt.suptitle('MCMC trace and marginal posteriors', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_traces.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# Corner plot (physical parameters only)
# ============================================================
try:
    import corner
    labels_corner = [r'd$\alpha$/dT (mm/yr/°C²)', r'$\alpha_0$ (mm/yr/°C)',
                     r'trend (mm/yr)']
    fig_corner = corner.corner(
        result.posterior_samples * M_TO_MM,
        labels=labels_corner,
        quantiles=[0.03, 0.50, 0.97],
        show_titles=True,
        title_fmt='.3f',
    )
    fig_corner.suptitle('Physical parameter posterior (mm/yr)', fontsize=14, y=1.02)
    plt.savefig(FIG_DIR / 'bayesian_level_corner.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('corner package not installed — skipping corner plot')

# ============================================================
# Autocorrelation
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i, name in enumerate(['dalpha_dT', 'alpha0', 'trend', 'log_sigma_extra', 'H0']):
    data = chain_full[name].values[0]  # first chain
    maxlag = min(200, len(data))
    acf = np.correlate(data - data.mean(), data - data.mean(), mode='full')
    acf = acf[len(acf)//2:]
    acf /= acf[0]
    axes[i].plot(acf[:maxlag], lw=0.8)
    axes[i].axhline(0, color='gray', ls='--', lw=0.5)
    axes[i].set_title(param_labels[i], fontsize=10)
    axes[i].set_xlabel('Lag')
    if i == 0:
        axes[i].set_ylabel('ACF')
plt.suptitle('Autocorrelation (first chain)', fontsize=12)
plt.tight_layout()
plt.show()

# Print convergence diagnostics
diag = result.sampler_diagnostics
print(f"Acceptance fraction: {diag['acceptance_fraction']:.3f}")
conv = diag.get('convergence', {})
if conv:
    print(f"R-hat max: {conv.get('rhat_max', 'N/A')}")
    print(f"ESS min:   {conv.get('ess_min', 'N/A')}")
    print(f"Converged: {conv.get('converged', 'N/A')}")

## 7. Comparison with Previous Calibrations

Compare the Bayesian level-space posterior with:
- **DOLS (frequentist level-space)**: Weighted least squares with HAC standard errors
- Both calibrated on the same Frederikse GMSL + Berkeley Earth temperature data

The DOLS point estimate sits on the collinearity ridge; the Bayesian posterior captures the full trade-off between $a$ and $b$ along that ridge, constrained by the Exponential PC prior on $a$ and half-normal prior on $b$.

In [ ]:
# ============================================================
# Run DOLS for comparison
# ============================================================
dols_result = calibrate_dols(
    df_frederikse['gmsl'],
    df_berkeley['temperature'],
    gmsl_sigma=df_frederikse['gmsl_sigma'],
    order=2,
    n_lags=2,
)
dols_c = dols_result.physical_coefficients * M_TO_MM   # [dα/dT, α₀, trend]
dols_se = dols_result.physical_se * M_TO_MM

# Bayesian posterior summary
bayes_c = result.physical_coefficients * M_TO_MM
bayes_hdi = result.physical_hdi_94 * M_TO_MM
bayes_median = np.median(result.posterior_samples, axis=0) * M_TO_MM

# ---- Comparison table ----
print('=' * 75)
print('Comparison: Bayesian Level-Space vs DOLS (mm/yr)')
print('=' * 75)
labels = ['dα/dT (mm/yr/°C²)', 'α₀ (mm/yr/°C)', 'trend (mm/yr)']
print(f'{"Parameter":<22} {"DOLS ± SE":>22} {"Bayes mean [94% HDI]":>30}')
print('-' * 75)
for k, lab in enumerate(labels):
    dols_str = f'{dols_c[k]:7.3f} ± {dols_se[k]:.3f}'
    bayes_str = f'{bayes_c[k]:7.3f}  [{bayes_hdi[k,0]:.3f}, {bayes_hdi[k,1]:.3f}]'
    print(f'{lab:<22} {dols_str:>22} {bayes_str:>30}')
print(f'\n{"R²":<22} {dols_result.r_squared:>22.4f} {result.r2:>30.4f}')

# ---- Forest plot ----
fig, ax = plt.subplots(figsize=(10, 4))
y_pos = np.array([0, 1, 2])
offset = 0.15

# DOLS
ax.errorbar(dols_c, y_pos - offset, xerr=1.96 * dols_se,
            fmt='s', color='C0', capsize=4, label='DOLS (freq.)', ms=7)
# Bayesian
ax.errorbar(bayes_c, y_pos + offset,
            xerr=[bayes_c - bayes_hdi[:, 0], bayes_hdi[:, 1] - bayes_c],
            fmt='o', color='C1', capsize=4, label='Bayesian level (94% HDI)', ms=7)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.axvline(0, color='gray', ls='--', lw=0.5)
ax.set_xlabel('Coefficient value (mm/yr or mm/yr/°C or mm/yr/°C²)')
ax.set_title('Coefficient comparison: Bayesian level-space vs DOLS')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_vs_dols.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 2D contour: a vs b (collinearity ridge) ----
fig, ax = plt.subplots(figsize=(7, 6))
a_samp = result.posterior_samples[:, 0] * M_TO_MM
b_samp = result.posterior_samples[:, 1] * M_TO_MM
ax.scatter(a_samp, b_samp, s=1, alpha=0.1, color='C1', rasterized=True)
ax.plot(dols_c[0], dols_c[1], 's', color='C0', ms=10, zorder=5,
        label='DOLS point estimate')
# DOLS confidence ellipse
from matplotlib.patches import Ellipse
cov_ab = dols_result.physical_covariance[:2, :2] * M_TO_MM**2
eigvals, eigvecs = np.linalg.eigh(cov_ab)
angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
for n_sig, alpha in [(1, 0.4), (2, 0.2)]:
    ell = Ellipse(xy=(dols_c[0], dols_c[1]),
                  width=2*n_sig*np.sqrt(eigvals[1]),
                  height=2*n_sig*np.sqrt(eigvals[0]),
                  angle=angle, fill=False, color='C0',
                  alpha=alpha, lw=1.5, ls='--')
    ax.add_patch(ell)
ax.set_xlabel(r'd$\alpha$/dT (mm/yr/°C²)')
ax.set_ylabel(r'$\alpha_0$ (mm/yr/°C)')
ax.set_title('Collinearity ridge: $a$ vs $b$ posterior')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_collinearity.png', dpi=150, bbox_inches='tight')
plt.show()

rho_ab = np.corrcoef(a_samp, b_samp)[0, 1]
print(f'\nPosterior corr(a, b) = {rho_ab:.3f}')

## 8. Model Fit and Residuals

Assess the quality of the Bayesian level-space fit by comparing observed and modeled GMSL, examining residual structure, and checking residual autocorrelation. A good model should yield white-noise residuals within the observation uncertainty envelope.

In [ ]:
# ============================================================
# Model fit: observed vs modeled GMSL
# ============================================================
sigma_extra_med = np.median(result.sigma_extra_posterior)
sigma_total = np.sqrt(sigma_obs_fixed**2 + sigma_extra_med**2)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 2, 1]})

# Panel 1: Observed vs modeled
ax = axes[0]
ax.fill_between(obs_years,
                (result.H_model_mean - 2*sigma_total) * M_TO_MM,
                (result.H_model_mean + 2*sigma_total) * M_TO_MM,
                alpha=0.15, color='C1', label=r'$\pm 2\sigma_{\rm total}$')
ax.plot(obs_years, result.H_obs * M_TO_MM, 'k.', ms=3, alpha=0.6,
        label='Observed GMSL')
ax.plot(obs_years, result.H_model_mean * M_TO_MM, 'C1-', lw=2,
        label='Bayesian posterior mean')

# Posterior predictive: 100 random draws
rng_pp = np.random.default_rng(123)
idx_pp = rng_pp.choice(len(result.posterior_samples), 100, replace=False)
for ii in idx_pp:
    a_i, b_i, c_i = result.posterior_samples[ii]
    H0_i = result.H0_posterior[ii]
    H_draw = a_i * I2_obs + b_i * I1_obs + c_i * I0_obs + H0_i
    ax.plot(obs_years, H_draw * M_TO_MM, 'C1-', alpha=0.02, lw=0.5)

ax.set_ylabel('GMSL (mm rel. 1995–2005)')
ax.set_title(f'Bayesian level-space fit  (R² = {result.r2:.4f})')
ax.legend(loc='upper left')

# Panel 2: Residuals
ax = axes[1]
resid_mm = result.residuals * M_TO_MM
ax.fill_between(obs_years, -2*sigma_total*M_TO_MM, 2*sigma_total*M_TO_MM,
                alpha=0.15, color='gray', label=r'$\pm 2\sigma_{\rm total}$')
ax.bar(obs_years, resid_mm, width=0.8, color='C0', alpha=0.6)
ax.axhline(0, color='k', ls='-', lw=0.5)
ax.set_ylabel('Residual (mm)')
ax.legend(loc='upper left')

# Panel 3: Residual ACF
ax = axes[2]
maxlag_r = min(30, len(resid_mm) - 1)
acf_r = np.correlate(resid_mm - resid_mm.mean(),
                     resid_mm - resid_mm.mean(), mode='full')
acf_r = acf_r[len(acf_r)//2:]
acf_r /= acf_r[0]
ax.bar(range(maxlag_r + 1), acf_r[:maxlag_r + 1], color='C0', alpha=0.6)
# 95% confidence band for white noise
ci_95 = 1.96 / np.sqrt(len(resid_mm))
ax.axhline(ci_95, color='red', ls='--', lw=0.8, label='95% CI (white noise)')
ax.axhline(-ci_95, color='red', ls='--', lw=0.8)
ax.set_xlabel('Lag (years)')
ax.set_ylabel('ACF')
ax.set_xlim(-0.5, maxlag_r + 0.5)
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_fit_residuals.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Residual statistics
print(f'Residual std:  {np.std(resid_mm):.2f} mm')
print(f'σ_extra (med): {sigma_extra_med*M_TO_MM:.2f} mm')
print(f'σ_total range: {sigma_total.min()*M_TO_MM:.1f}–'
      f'{sigma_total.max()*M_TO_MM:.1f} mm')
n_outside_2sig = np.sum(np.abs(resid_mm) > 2*sigma_total*M_TO_MM)
print(f'Points outside ±2σ: {n_outside_2sig}/{len(resid_mm)} '
      f'({100*n_outside_2sig/len(resid_mm):.1f}%, expected ~5%)')

## 9. $\sigma_{\rm extra}$ Interpretation

The learned model-inadequacy parameter $\sigma_{\rm extra}$ captures unmodeled variability (ENSO, volcanic aerosols, internal climate modes) beyond the formal observation uncertainty. Its posterior should be consistent with the magnitude of these known sources of interannual GMSL variability.

In [ ]:
# ============================================================
# σ_extra posterior analysis
# ============================================================
se_mm = result.sigma_extra_posterior * M_TO_MM

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax = axes[0]
ax.hist(se_mm, bins=60, density=True, alpha=0.6, color='C2',
        edgecolor='C2', lw=0.5)
ax.axvline(np.median(se_mm), color='k', ls='-', lw=2,
           label=f'Median = {np.median(se_mm):.2f} mm')
ax.axvline(np.percentile(se_mm, 3), color='k', ls='--', lw=1,
           label=f'94% HDI: [{np.percentile(se_mm, 3):.2f}, '
                 f'{np.percentile(se_mm, 97):.2f}]')
ax.axvline(np.percentile(se_mm, 97), color='k', ls='--', lw=1)
ax.set_xlabel(r'$\sigma_{\rm extra}$ (mm)')
ax.set_ylabel('Density')
ax.set_title(r'Posterior of $\sigma_{\rm extra}$')
ax.legend(fontsize=9)

# Compare σ_extra to formal uncertainty
ax = axes[1]
ax.plot(obs_years, sigma_gmsl * M_TO_MM, label=r'$\sigma_{\rm gmsl}$', lw=1.5)
ax.plot(obs_years, sigma_tws_total * M_TO_MM, label=r'$\sigma_{\rm TWS}$', lw=1.5)
ax.axhline(np.median(se_mm), color='C2', ls='-', lw=2,
           label=rf'$\sigma_{{\rm extra}}$ = {np.median(se_mm):.1f} mm')
ax.fill_between(obs_years,
                np.percentile(se_mm, 3) * np.ones_like(obs_years),
                np.percentile(se_mm, 97) * np.ones_like(obs_years),
                alpha=0.2, color='C2')
ax.plot(obs_years, sigma_total * M_TO_MM, 'k-', lw=2,
        label=r'$\sigma_{\rm total}$ (incl $\sigma_{\rm extra}$)')
ax.set_xlabel('Year')
ax.set_ylabel('Uncertainty (mm)')
ax.set_title('Uncertainty budget with learned model inadequacy')
ax.legend(fontsize=9)
ax.set_xlim(1900, 2020)
ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_sigma_extra.png', dpi=150,
            bbox_inches='tight')
plt.show()

# ---- Coverage check ----
# Use all posterior draws to compute empirical coverage
coverages = {}
for pct in [50, 68, 90, 95]:
    count_in = 0
    for ii in range(min(1000, len(result.posterior_samples))):
        a_i, b_i, c_i = result.posterior_samples[ii]
        H0_i = result.H0_posterior[ii]
        se_i = result.sigma_extra_posterior[ii]
        H_pred = a_i * I2_obs + b_i * I1_obs + c_i * I0_obs + H0_i
        sig_i = np.sqrt(sigma_obs_fixed**2 + se_i**2)
        z_crit = {50: 0.674, 68: 1.0, 90: 1.645, 95: 1.96}[pct]
        in_band = np.abs(result.H_obs - H_pred) <= z_crit * sig_i
        count_in += in_band.mean()
    coverages[pct] = count_in / min(1000, len(result.posterior_samples))

print('Empirical coverage (posterior predictive):')
for pct, cov in coverages.items():
    print(f'  {pct}% CI: actual coverage = {100*cov:.1f}%')

## 10. Forward Projections

Use the posterior coefficient samples to project GMSL under SSP temperature scenarios via `project_gmsl_ensemble()`. The posterior samples preserve the full non-Gaussian structure (Exponential PC prior on $a$, half-normal on $b$, collinearity ridge) rather than assuming a multivariate normal approximation.

In [ ]:
# ============================================================
# Forward projections using posterior samples
# ============================================================
# Bayesian level-space projections
mc_bayes = project_gmsl_ensemble(
    coefficients=result.physical_coefficients,
    coefficients_cov=result.physical_covariance,
    temperature_projections=temp_projections,
    baseline_year=BASELINE_YEAR,
    baseline_gmsl=0.0,
    n_samples=2000,
    posterior_samples=result.posterior_samples,
    seed=42,
)

# DOLS projections for comparison
mc_dols = project_gmsl_ensemble(
    coefficients=dols_result.physical_coefficients,
    coefficients_cov=dols_result.physical_covariance,
    temperature_projections=temp_projections,
    baseline_year=BASELINE_YEAR,
    baseline_gmsl=0.0,
    n_samples=2000,
    seed=42,
)

# Summary at key years
print('=' * 80)
print('Projection summary at 2050, 2100, 2150 (mm rel. 2005)')
print('=' * 80)
for yr in [2050, 2100, 2150]:
    print(f'\n--- Year {yr} ---')
    print(f'{"SSP":<12} {"Bayesian med [5–95%]":>28} {"DOLS med [5–95%]":>28}')
    print('-' * 70)
    for ssp in ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']:
        t_b = mc_bayes[ssp]['time']
        idx_b = np.argmin(np.abs(t_b - yr))
        samp_b = mc_bayes[ssp]['gmsl_samples'][:, idx_b] * M_TO_MM
        b_str = (f'{np.median(samp_b):6.0f}  '
                 f'[{np.percentile(samp_b, 5):5.0f}, '
                 f'{np.percentile(samp_b, 95):5.0f}]')

        t_d = mc_dols[ssp]['time']
        idx_d = np.argmin(np.abs(t_d - yr))
        samp_d = mc_dols[ssp]['gmsl_samples'][:, idx_d] * M_TO_MM
        d_str = (f'{np.median(samp_d):6.0f}  '
                 f'[{np.percentile(samp_d, 5):5.0f}, '
                 f'{np.percentile(samp_d, 95):5.0f}]')

        print(f'{ssp:<12} {b_str:>28} {d_str:>28}')

## 11. Projection Comparison

Compare Bayesian level-space, DOLS, and IPCC AR6 projections across SSP scenarios. The key question is whether the different calibration approaches — despite sitting at different locations on the collinearity ridge — produce similar projection uncertainty envelopes.

In [ ]:
# ============================================================
# Multi-panel projection comparison
# ============================================================
ssp_colors = {
    'SSP1-2.6': 'C0', 'SSP2-4.5': 'C1',
    'SSP3-7.0': 'C3', 'SSP5-8.5': 'C4',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, ssp in zip(axes.flat, ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']):
    color = ssp_colors[ssp]

    # Bayesian level-space
    t_b = mc_bayes[ssp]['time']
    med_b = mc_bayes[ssp]['gmsl_median'] * M_TO_MM
    p5_b = mc_bayes[ssp]['gmsl_p5'] * M_TO_MM
    p95_b = mc_bayes[ssp]['gmsl_p95'] * M_TO_MM
    ax.fill_between(t_b, p5_b, p95_b, alpha=0.15, color=color)
    ax.plot(t_b, med_b, '-', color=color, lw=2, label='Bayesian level')

    # DOLS
    t_d = mc_dols[ssp]['time']
    med_d = mc_dols[ssp]['gmsl_median'] * M_TO_MM
    p5_d = mc_dols[ssp]['gmsl_p5'] * M_TO_MM
    p95_d = mc_dols[ssp]['gmsl_p95'] * M_TO_MM
    ax.plot(t_d, med_d, '--', color=color, lw=1.5, alpha=0.7, label='DOLS')
    ax.fill_between(t_d, p5_d, p95_d, alpha=0.08, color=color,
                    linestyle='--', edgecolor=color, lw=0.5)

    # IPCC AR6
    if ssp in ipcc_gmsl:
        ipcc = ipcc_gmsl[ssp]
        ipcc_t = np.array([
            t.year + (t.month - 0.5) / 12 for t in ipcc.index
        ])
        # IPCC data has gmsl, gmsl_lower, gmsl_upper (meters)
        cols = ipcc.columns
        if 'gmsl' in cols:
            ax.plot(ipcc_t, ipcc['gmsl'].values * M_TO_MM, ':',
                    color='gray', lw=2, label='IPCC AR6')
            if 'gmsl_lower' in cols and 'gmsl_upper' in cols:
                ax.fill_between(
                    ipcc_t,
                    ipcc['gmsl_lower'].values * M_TO_MM,
                    ipcc['gmsl_upper'].values * M_TO_MM,
                    alpha=0.1, color='gray',
                )

    ax.set_title(ssp, fontsize=13)
    ax.axhline(0, color='gray', ls=':', lw=0.5)
    if ax in axes[1]:
        ax.set_xlabel('Year')
    if ax in axes[:, 0]:
        ax.set_ylabel('GMSL (mm rel. 2005)')

axes[0, 0].legend(fontsize=9, loc='upper left')
axes[0, 0].set_xlim(2000, 2150)
plt.suptitle('GMSL projections: Bayesian level-space vs DOLS vs IPCC AR6',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_projections.png', dpi=150,
            bbox_inches='tight')
plt.show()

# ---- Bar chart at 2100 ----
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ssp_colors))
width = 0.25

for i, ssp in enumerate(ssp_colors):
    t_b = mc_bayes[ssp]['time']
    idx_100 = np.argmin(np.abs(t_b - 2100))
    samp_b = mc_bayes[ssp]['gmsl_samples'][:, idx_100] * M_TO_MM
    ax.bar(i - width, np.median(samp_b), width,
           yerr=[[np.median(samp_b) - np.percentile(samp_b, 5)],
                 [np.percentile(samp_b, 95) - np.median(samp_b)]],
           capsize=3, color=ssp_colors[ssp], alpha=0.7,
           label='Bayesian' if i == 0 else '')

    t_d = mc_dols[ssp]['time']
    idx_100d = np.argmin(np.abs(t_d - 2100))
    samp_d = mc_dols[ssp]['gmsl_samples'][:, idx_100d] * M_TO_MM
    ax.bar(i, np.median(samp_d), width,
           yerr=[[np.median(samp_d) - np.percentile(samp_d, 5)],
                 [np.percentile(samp_d, 95) - np.median(samp_d)]],
           capsize=3, color=ssp_colors[ssp], alpha=0.4,
           label='DOLS' if i == 0 else '', hatch='//')

    if ssp in ipcc_gmsl:
        ipcc = ipcc_gmsl[ssp]
        ipcc_t = np.array([
            t.year + (t.month - 0.5) / 12 for t in ipcc.index
        ])
        idx_ipcc = np.argmin(np.abs(ipcc_t - 2100))
        if 'gmsl' in ipcc.columns:
            ipcc_med = ipcc['gmsl'].iloc[idx_ipcc] * M_TO_MM
            ax.bar(i + width, ipcc_med, width,
                   color='gray', alpha=0.4,
                   label='IPCC AR6' if i == 0 else '')

ax.set_xticks(x)
ax.set_xticklabels(list(ssp_colors.keys()))
ax.set_ylabel('GMSL at 2100 (mm rel. 2005)')
ax.set_title('GMSL at 2100: Bayesian level vs DOLS vs IPCC AR6')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_2100_comparison.png', dpi=150,
            bbox_inches='tight')
plt.show()

## 12. Sensitivity Analysis

Test robustness of the posterior to:
1. **Prior width**: 0.5× and 2× the default prior scale for $a$ and $b$
2. **Dam-filling amplitude**: 0, 3, 6 mm structural TWS uncertainty
3. **$\sigma_{\rm extra}$ prior**: HalfCauchy(5mm) vs HalfCauchy(10mm)

If the posterior is dominated by the likelihood (data-informed), these variations should have minor impact. If the prior is influential (data-limited), sensitivity will be larger — indicating which parameters are most constrained by prior assumptions vs observations.

In [ ]:
# ============================================================
# Sensitivity analysis
# ============================================================
# Use fewer samples for sensitivity runs (speed)
N_SENS = 1500
N_BURN_SENS = 500

sensitivity_results = {}

# ---- 1. Prior width sensitivity ----
# Vary Exponential mean for a (PC prior) and HalfNormal sigma for b
prior_configs = {
    'strong (0.5\u00d7)': dict(
        prior_scale_a=calibrate_exponential_prior_a(0.05, 0.005),  # P(a>5)=5%
        prior_scale_b=0.005),
    'default': dict(
        prior_scale_a=PRIOR_SCALE_A,
        prior_scale_b=PRIOR_SCALE_B),
    'weak (2\u00d7)': dict(
        prior_scale_a=calibrate_exponential_prior_a(0.20, 0.005),  # P(a>5)=20%
        prior_scale_b=0.020),
}
print('=== Prior width sensitivity ===')
for label, kw in prior_configs.items():
    print(f'  {label}: \u03bc_a={kw["prior_scale_a"]*M_TO_MM:.2f}, '
          f'\u03c3_b={kw["prior_scale_b"]*M_TO_MM:.1f} mm/yr')
    if label == 'default':
        sensitivity_results[label] = result
        print(f'    -> using existing result')
        continue
    r = fit_bayesian_level(
        H_obs=H_obs_full, sigma_obs=sigma_obs_fixed,
        I2_obs=I2_obs, I1_obs=I1_obs, I0_obs=I0_obs,
        prior_c_mean=PRIOR_C_MEAN, prior_c_sigma=PRIOR_C_SIGMA,
        prior_sigma_extra_scale=PRIOR_SIGMA_EXTRA_SCALE,
        prior_H0_sigma=PRIOR_H0_SIGMA,
        n_samples=N_SENS, n_walkers=N_WALKERS, n_burnin=N_BURN_SENS,
        progress=False, seed=42, **kw,
    )
    sensitivity_results[label] = r
    c_mm = r.physical_coefficients * M_TO_MM
    print(f'    -> a={c_mm[0]:.3f}, b={c_mm[1]:.3f}, '
          f'c={c_mm[2]:.3f} mm/yr, R\u00b2={r.r2:.4f}')

# ---- 2. Dam-filling amplitude sensitivity ----
dam_configs = {
    'no dam (0 mm)': 0.000,
    'default (3 mm)': 0.003,
    'large (6 mm)': 0.006,
}
print('\n=== Dam-filling amplitude sensitivity ===')
for label, amp in dam_configs.items():
    dam_e = amp * np.exp(-0.5 * ((obs_years - DAM_PEAK) / DAM_WIDTH)**2)
    sig_tws = np.sqrt(tws_sigma**2 + dam_e**2)
    sig_obs_v = np.sqrt(sigma_gmsl**2 + sig_tws**2)

    if label == 'default (3 mm)':
        sensitivity_results[f'dam: {label}'] = result
        print(f'  {label}: using existing result')
        continue

    r = fit_bayesian_level(
        H_obs=H_obs_full, sigma_obs=sig_obs_v,
        I2_obs=I2_obs, I1_obs=I1_obs, I0_obs=I0_obs,
        prior_scale_a=PRIOR_SCALE_A, prior_scale_b=PRIOR_SCALE_B,
        prior_c_mean=PRIOR_C_MEAN, prior_c_sigma=PRIOR_C_SIGMA,
        prior_sigma_extra_scale=PRIOR_SIGMA_EXTRA_SCALE,
        prior_H0_sigma=PRIOR_H0_SIGMA,
        n_samples=N_SENS, n_walkers=N_WALKERS, n_burnin=N_BURN_SENS,
        progress=False, seed=42,
    )
    sensitivity_results[f'dam: {label}'] = r
    c_mm = r.physical_coefficients * M_TO_MM
    print(f'  {label}: a={c_mm[0]:.3f}, b={c_mm[1]:.3f}, '
          f'c={c_mm[2]:.3f} mm/yr, \u03c3_extra={np.median(r.sigma_extra_posterior)*M_TO_MM:.2f} mm')

# ---- 3. \u03c3_extra prior sensitivity ----
se_configs = {
    'HC(5mm)': 0.005,
    'HC(10mm)': 0.010,
}
print('\n=== \u03c3_extra prior sensitivity ===')
for label, scale in se_configs.items():
    if label == 'HC(5mm)':
        sensitivity_results[f'\u03c3_extra: {label}'] = result
        print(f'  {label}: using existing result')
        continue
    r = fit_bayesian_level(
        H_obs=H_obs_full, sigma_obs=sigma_obs_fixed,
        I2_obs=I2_obs, I1_obs=I1_obs, I0_obs=I0_obs,
        prior_scale_a=PRIOR_SCALE_A, prior_scale_b=PRIOR_SCALE_B,
        prior_c_mean=PRIOR_C_MEAN, prior_c_sigma=PRIOR_C_SIGMA,
        prior_sigma_extra_scale=scale,
        prior_H0_sigma=PRIOR_H0_SIGMA,
        n_samples=N_SENS, n_walkers=N_WALKERS, n_burnin=N_BURN_SENS,
        progress=False, seed=42,
    )
    sensitivity_results[f'\u03c3_extra: {label}'] = r
    c_mm = r.physical_coefficients * M_TO_MM
    print(f'  {label}: a={c_mm[0]:.3f}, b={c_mm[1]:.3f}, '
          f'c={c_mm[2]:.3f} mm/yr, \u03c3_extra={np.median(r.sigma_extra_posterior)*M_TO_MM:.2f} mm')

# ---- Summary plot ----
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
param_names_plot = [r'd$\alpha$/dT', r'$\alpha_0$', 'trend']

# Prior width
ax = axes[0]
for j, (label, col) in enumerate(zip(prior_configs.keys(), ['C0', 'C1', 'C2'])):
    r = sensitivity_results.get(label, sensitivity_results.get(f'dam: {label}'))
    if r is None:
        continue
    for k in range(3):
        samp = r.posterior_samples[:, k] * M_TO_MM
        parts = ax.violinplot([samp], positions=[k + j*0.25 - 0.25],
                              widths=0.2, showmedians=True)
        for pc in parts['bodies']:
            pc.set_facecolor(col)
            pc.set_alpha(0.4)
        for part in ['cmins', 'cmaxes', 'cmedians', 'cbars']:
            if part in parts:
                parts[part].set_color(col)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(param_names_plot)
ax.set_ylabel('mm/yr (or mm/yr/\u00b0C or mm/yr/\u00b0C\u00b2)')
ax.set_title('Prior width sensitivity')

# Dam amplitude
ax = axes[1]
dam_labels = list(dam_configs.keys())
for j, label in enumerate(dam_labels):
    key = f'dam: {label}' if f'dam: {label}' in sensitivity_results else label
    r = sensitivity_results.get(key)
    if r is None:
        continue
    se_vals = r.sigma_extra_posterior * M_TO_MM
    ax.hist(se_vals, bins=40, alpha=0.4, density=True, label=label)
ax.set_xlabel(r'$\sigma_{\rm extra}$ (mm)')
ax.set_ylabel('Density')
ax.set_title(r'Dam amplitude $\rightarrow$ $\sigma_{\rm extra}$')
ax.legend(fontsize=9)

# \u03c3_extra prior
ax = axes[2]
for label in ['HC(5mm)', 'HC(10mm)']:
    key = f'\u03c3_extra: {label}'
    r = sensitivity_results.get(key)
    if r is None:
        continue
    se_vals = r.sigma_extra_posterior * M_TO_MM
    ax.hist(se_vals, bins=40, alpha=0.4, density=True, label=label)
ax.set_xlabel(r'$\sigma_{\rm extra}$ (mm)')
ax.set_ylabel('Density')
ax.set_title(r'$\sigma_{\rm extra}$ prior sensitivity')
ax.legend(fontsize=9)

plt.suptitle('Sensitivity analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'bayesian_level_sensitivity.png', dpi=150,
            bbox_inches='tight')
plt.show()

## 13. Summary

Export key results in JSON format compatible with the project's `results_summary.json` structure.

In [ ]:
# ============================================================
# Export results to JSON
# ============================================================
import json

bayes_summary = {
    'model': 'bayesian_level_space',
    'description': 'Self-consistent Bayesian level-space calibration',
    'calibration': {
        'n_obs': len(H_obs_full),
        'obs_range': [float(obs_years[0]), float(obs_years[-1])],
        'r_squared': float(result.r2),
        'coefficients_mm_yr': {
            'dalpha_dT': float(result.physical_coefficients[0] * M_TO_MM),
            'alpha0': float(result.physical_coefficients[1] * M_TO_MM),
            'trend': float(result.physical_coefficients[2] * M_TO_MM),
        },
        'hdi_94_mm_yr': {
            'dalpha_dT': [float(x) for x in result.physical_hdi_94[0] * M_TO_MM],
            'alpha0': [float(x) for x in result.physical_hdi_94[1] * M_TO_MM],
            'trend': [float(x) for x in result.physical_hdi_94[2] * M_TO_MM],
        },
        'sigma_extra_mm': {
            'median': float(np.median(result.sigma_extra_posterior) * M_TO_MM),
            'hdi_94': [
                float(np.percentile(result.sigma_extra_posterior, 3) * M_TO_MM),
                float(np.percentile(result.sigma_extra_posterior, 97) * M_TO_MM),
            ],
        },
    },
    'priors': {
        'a': f'Exponential(mean={PRIOR_SCALE_A*M_TO_MM:.2f} mm/yr/C^2)',
        'b': f'HalfNormal(sigma={PRIOR_SCALE_B*M_TO_MM:.1f} mm/yr/C)',
        'c': f'Normal({PRIOR_C_MEAN*M_TO_MM:.1f}, {PRIOR_C_SIGMA*M_TO_MM:.1f} mm/yr)',
        'sigma_extra': f'HalfCauchy(0, {PRIOR_SIGMA_EXTRA_SCALE*M_TO_MM:.1f} mm)',
        'H0': f'Normal(H_obs[0], {PRIOR_H0_SIGMA*M_TO_MM:.1f} mm)',
    },
    'uncertainty_model': {
        'dam_peak_year': DAM_PEAK,
        'dam_width_yr': DAM_WIDTH,
        'dam_amplitude_mm': DAM_AMPLITUDE * M_TO_MM,
    },
    'sampler': {
        'n_walkers': N_WALKERS,
        'n_samples': N_SAMPLES,
        'n_burnin': N_BURNIN,
        'acceptance_fraction': float(
            result.sampler_diagnostics['acceptance_fraction']),
    },
    'projections_at_2100_mm': {},
}

# Add projection results at 2100
for ssp in ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']:
    t_b = mc_bayes[ssp]['time']
    idx = np.argmin(np.abs(t_b - 2100))
    samp = mc_bayes[ssp]['gmsl_samples'][:, idx] * M_TO_MM
    bayes_summary['projections_at_2100_mm'][ssp] = {
        'median': float(np.median(samp)),
        'p5': float(np.percentile(samp, 5)),
        'p95': float(np.percentile(samp, 95)),
        'p17': float(np.percentile(samp, 17)),
        'p83': float(np.percentile(samp, 83)),
    }

# Save
out_path = DATA_DIR / 'processed' / 'bayesian_level_results.json'
with open(out_path, 'w') as f:
    json.dump(bayes_summary, f, indent=2)
print(f'Results saved to {out_path}')

# Print summary
print('\n' + '=' * 70)
print('BAYESIAN LEVEL-SPACE MODEL \u2014 SUMMARY')
print('=' * 70)
cal = bayes_summary['calibration']
print(f"\nCalibration (Frederikse GMSL, {cal['obs_range'][0]:.0f}"
      f"\u2013{cal['obs_range'][1]:.0f}, n={cal['n_obs']}):")
print(f"  R\u00b2 = {cal['r_squared']:.4f}")
for name in ['dalpha_dT', 'alpha0', 'trend']:
    v = cal['coefficients_mm_yr'][name]
    hdi = cal['hdi_94_mm_yr'][name]
    print(f'  {name}: {v:.3f} [{hdi[0]:.3f}, {hdi[1]:.3f}] mm/yr')
se = cal['sigma_extra_mm']
print(f"  \u03c3_extra: {se['median']:.2f} [{se['hdi_94'][0]:.2f}, "
      f"{se['hdi_94'][1]:.2f}] mm")

print(f'\nProjections at 2100 (mm, relative to 2005):')
for ssp, p in bayes_summary['projections_at_2100_mm'].items():
    print(f"  {ssp}: {p['median']:.0f} [{p['p5']:.0f}, {p['p95']:.0f}]")